# Lab 1: Conditional Probability and Bayes' Rule

**Phân tích dữ liệu Bayesian - IUH**

Một con xúc xắc, một viên bi đỏ, một kết quả xét nghiệm dương tính. Nhìn bề ngoài, đó là những bài toán rất khác nhau. Nhưng nếu nhìn kỹ hơn, chúng đều hỏi cùng một điều: sau khi biết thêm một mẩu thông tin mới, ta phải thay đổi cách nhìn về thế giới ra sao?

Lab 1 chính là nơi câu hỏi ấy được đặt ở dạng đơn giản nhất. Có lúc thông tin mới chỉ làm ta thu hẹp không gian mẫu. Có lúc nó buộc ta đi ngược từ dữ liệu quan sát được về nguyên nhân ẩn phía sau. Và chính từ những bài toán sơ cấp này, trực giác Bayesian bắt đầu hình thành.


In [ ]:
# Các thư viện ở đây hỗ trợ cả tính toán xác suất rời rạc lẫn minh họa nhanh nếu cần.
import math
from fractions import Fraction
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

plt.rcParams["figure.figsize"] = (10, 5)


## 1. Khi thông tin làm đổi bài toán

Ở những bài đầu của lab, điều quan trọng không phải là những con số cuối cùng mà là cách đọc đúng biến cố điều kiện. Biết tổng hai lần tung xúc xắc bằng 7 không giống với việc biết riêng từng lần tung. Biết “ít nhất một con là nữ” không giống với việc chọn ngẫu nhiên một người con rồi phát hiện đó là nữ. Mỗi cách nhận thông tin đều tạo ra một không gian mẫu điều kiện khác nhau.

Đến các bài túi bi và xét nghiệm y khoa, câu chuyện tiến thêm một bước: dữ liệu quan sát được không còn chỉ là thông tin phụ, mà trở thành dấu vết để suy ra nguyên nhân ẩn. Khi đó, công thức Bayes không còn là một công thức trang trí, mà là ngôn ngữ tự nhiên để cập nhật niềm tin từ bằng chứng.


## 2. Bài 1 - Xúc xắc và xác suất có điều kiện

### Problem framing

Ở phần (a), ta không suy luận về một tham số ẩn mà chỉ thu hẹp không gian mẫu sau khi biết biến cố điều kiện. Ở phần (b), thông tin $X+Y=7$ loại bỏ hầu hết các cặp kết quả và làm xác suất điều kiện trở thành bài toán đếm trên một tập con nhỏ hơn.

### Ý nghĩa của bài tập

Bài này dạy một kỹ năng nền tảng: thông tin điều kiện không thay đổi bản chất phép thử, nhưng nó thay đổi tập kết quả còn khả dĩ. Vì vậy, xác suất điều kiện không nên được học như một công thức máy móc mà như thao tác “lọc lại” không gian mẫu sau khi biết thêm dữ liệu.

Đây cũng là bước đệm rất quan trọng cho tư duy Bayesian. Ở mức trừu tượng hơn, posterior chính là một phiên bản giàu cấu trúc hơn của cùng ý tưởng này: niềm tin ban đầu được thu hẹp lại sau khi ta biết thêm thông tin quan sát được.


### Đề bài nhắc lại

**Bài 1.**

- a) Tung một con xúc xắc đồng chất. Gọi $A$ là biến cố mặt thu được là lẻ, $B$ là biến cố mặt thu được không vượt quá $3$. Tính $P(A\mid B)$ và $P(B\mid A)$.
- b) Tung một con xúc xắc đồng chất $2$ lần. Gọi $X, Y$ là kết quả ở lần tung thứ nhất và thứ hai. Tính $P((X=3) \vee (Y=3) \mid X+Y=7)$.


In [ ]:
# Liệt kê trực tiếp không gian mẫu để biến xác suất có điều kiện thành bài toán đếm minh bạch.
omega = set(range(1, 7))
A = {1, 3, 5}
B = {1, 2, 3}
p_A_given_B = Fraction(len(A & B), len(B))
p_B_given_A = Fraction(len(A & B), len(A))

pairs = list(product(range(1, 7), repeat=2))
condition = [(x, y) for x, y in pairs if x + y == 7]
event = [(x, y) for x, y in condition if x == 3 or y == 3]
p_event_given_condition = Fraction(len(event), len(condition))

print(f"P(A|B) = {p_A_given_B}")
print(f"P(B|A) = {p_B_given_A}")
print(f"P((X=3) or (Y=3) | X+Y=7) = {p_event_given_condition}")


### Interpretation

Kết quả cho thấy $P(A\mid B)=\frac{2}{3}$ còn $P(B\mid A)=\frac{2}{3}$ trong bài này là trùng nhau do hình dạng đặc biệt của hai tập biến cố, chứ không phải là quy luật chung. Ở phần (b), khi biết $X+Y=7$ thì chỉ còn 6 cặp khả dĩ, trong đó có 2 cặp chứa số 3, nên xác suất điều kiện bằng $\frac{1}{3}$.


## 3. Bài 2 - Túi bi và Bayes rule

Ta chọn ngẫu nhiên một túi rồi chọn một viên bi. Màu viên bi là dữ liệu quan sát, còn chiếc túi được chọn là nguyên nhân ẩn. Đây là bối cảnh cổ điển để dùng Bayes: posterior trả lời câu hỏi “nếu đã thấy bi đỏ thì khả năng nó đến từ túi nào là bao nhiêu?”.

### Ý nghĩa của bài tập

Bài túi bi là một ví dụ Bayes mẫu mực vì nó phân tách rất rõ giữa nguyên nhân ẩn và dữ liệu quan sát. Ta không nhìn thấy chiếc túi nào đã được chọn, nhưng ta thấy màu viên bi. Công thức Bayes cho phép đi ngược từ dấu vết quan sát được về nguyên nhân có khả năng sinh ra dấu vết đó.

Ý nghĩa sư phạm của bài này là giúp sinh viên thấy Bayes không chỉ là một công thức đổi vế. Nó là cách kết hợp prior về nguyên nhân với mức độ phù hợp của dữ liệu dưới từng giả thuyết để tạo ra posterior.


### Đề bài nhắc lại

**Bài 2.** Có ba túi, mỗi túi chứa $100$ viên bi. Túi 1 có $75$ đỏ và $25$ xanh, túi 2 có $60$ đỏ và $40$ xanh, túi 3 có $45$ đỏ và $55$ xanh. Chọn ngẫu nhiên một túi, rồi bốc ngẫu nhiên một viên bi từ túi đó.

- a) Tính xác suất viên bi bốc ra là màu đỏ.
- b) Biết rằng viên bi bốc ra là đỏ, tính xác suất túi được chọn là túi 1.


In [ ]:
# Bài túi bi: màu viên bi là dữ liệu quan sát, còn chiếc túi là nguyên nhân ẩn cần suy luận bằng Bayes.
bag_prior = np.array([1/3, 1/3, 1/3])
p_red_given_bag = np.array([0.75, 0.60, 0.45])
p_red = np.dot(bag_prior, p_red_given_bag)
posterior_bag1_given_red = bag_prior[0] * p_red_given_bag[0] / p_red

print(f"P(red) = {p_red:.4f}")
print(f"P(bag 1 | red) = {posterior_bag1_given_red:.4f}")


### Interpretation

Trước khi quan sát gì, mỗi túi có xác suất $\frac{1}{3}$. Sau khi thấy bi đỏ, posterior nghiêng về túi 1 vì túi này có tỷ lệ bi đỏ cao nhất. Ta thu được $P(\text{đỏ})=0.6$ và $P(\text{túi 1}\mid \text{đỏ})=\frac{5}{12}\approx 0.4167$.


## 4. Bài 3 - Bài toán hai con và cấu trúc thông tin

Điểm cốt lõi của bài này là cách thông tin được cung cấp quyết định posterior. Câu “con đầu là nữ”, “có ít nhất một con là nữ” và “chọn ngẫu nhiên một người con rồi biết đó là nữ” không tạo ra cùng một không gian điều kiện.

### Ý nghĩa của bài tập

Bài toán hai con nổi tiếng vì nó cho thấy posterior phụ thuộc vào cách thông tin được tạo ra và tiết lộ, chứ không chỉ phụ thuộc vào nội dung câu chữ bề mặt. Hai phát biểu nghe gần giống nhau có thể dẫn đến hai không gian mẫu điều kiện hoàn toàn khác nhau.

Đây là bài học rất quan trọng cho thống kê hiện đại: trước khi tính toán, ta phải hỏi dữ liệu đến với ta bằng cơ chế nào. Nếu mô tả cơ chế lấy mẫu hoặc cơ chế báo tin bị mơ hồ, kết quả xác suất sẽ sai ngay từ gốc.


### Đề bài nhắc lại

**Bài 3.** Xét gia đình có hai con với không gian mẫu giới tính $\Omega = \{(G,G),(G,B),(B,G),(B,B)\}$ và giả sử mọi khả năng là như nhau.

- a) Tính xác suất cả hai đều là nữ, biết rằng đứa con đầu là nữ.
- b) Biết rằng gia đình có ít nhất một con là nữ, tính xác suất cả hai đều là nữ.
- c) Chọn ngẫu nhiên một người con trong gia đình và biết người được chọn là nữ; tính xác suất cả hai đều là nữ.
- d) Nếu xác suất một bé gái được đặt tên *Thanh Thảo* là $\alpha$ và độc lập với tên đứa còn lại, biết rằng gia đình có ít nhất một con tên *Thanh Thảo*, tính xác suất cả hai đều là nữ.


In [ ]:
# Bài hai con: cùng một dữ kiện về “bé gái” nhưng cơ chế cung cấp thông tin khác nhau sẽ cho posterior khác nhau.
families = [("G", "G"), ("G", "B"), ("B", "G"), ("B", "B")]

# a) first child is girl
case_a = [fam for fam in families if fam[0] == "G"]
# b) at least one girl
case_b = [fam for fam in families if "G" in fam]
# c) choose a random child and are told that child is girl
weighted_c = {
    ("G", "G"): 1/4 * 1,
    ("G", "B"): 1/4 * 1/2,
    ("B", "G"): 1/4 * 1/2,
}
prob_c = weighted_c[("G", "G")] / sum(weighted_c.values())

alpha = 0.02
prob_named = ((2 * alpha - alpha**2) / 4) / (((2 * alpha - alpha**2) / 4) + alpha/4 + alpha/4)

print(f"a) P(two girls | first is girl) = {Fraction(1,2)}")
print(f"b) P(two girls | at least one girl) = {Fraction(1,3)}")
print(f"c) P(two girls | randomly selected child is girl) = {prob_c:.4f}")
print(f"d) With alpha={alpha}, P(two girls | at least one named Thanh Thao) = {prob_named:.4f}")
print("General formula for part d:")
print("P = (2α - α²) / (4α - α²) = (2 - α) / (4 - α)")


### Interpretation

Phần (a) cho xác suất $\frac{1}{2}$ vì khi biết con đầu là nữ, chỉ còn hai gia đình khả dĩ. Phần (b) cho $\frac{1}{3}$ vì điều kiện “ít nhất một con là nữ” vẫn giữ lại ba khả năng. Phần (c) quay lại gần $\frac{1}{2}$ vì cơ chế quan sát ngẫu nhiên ưu tiên gia đình có hai con gái. Phần (d) cho công thức $\frac{2-\alpha}{4-\alpha}$, và khi $\alpha\approx 0$ thì xác suất tiến gần $\frac{1}{2}$.


## 5. Bài 4 - Chain rule và lấy mẫu không hoàn lại

Công thức
$$P(A \cap B \cap C)=P(A)P(B\mid A)P(C\mid A,B)$$
là trường hợp riêng của quy tắc chuỗi tổng quát
$$P(A_1\cap\cdots\cap A_k)=P(A_1)\prod_{j=2}^k P(A_j\mid A_1,\ldots,A_{j-1}).$$

Bài toán nhà máy chỉ là một ứng dụng trực tiếp: xác suất 3 lần liên tiếp không bốc trúng hàng lỗi bằng tích các xác suất điều kiện ở từng bước.

### Ý nghĩa của bài tập

Bài này giúp sinh viên nhìn xác suất giao của nhiều biến cố như một quá trình tuần tự. Thay vì cố nhớ những công thức riêng lẻ, ta có thể đọc bài toán theo dòng thời gian: sau bước đầu tiên, xác suất ở bước tiếp theo phải được cập nhật theo những gì đã xảy ra.

Ý tưởng này là cây cầu nối rất tự nhiên sang Bayesian updating. Trong cả hai trường hợp, ta đều tích lũy thông tin từng bước và điều chỉnh các xác suất điều kiện cho phù hợp với lịch sử quan sát.


### Đề bài nhắc lại

**Bài 4.**

- Chứng minh công thức $P(A \cap B \cap C) = P(A)P(B\mid A)P(C\mid A,B)$ và phát biểu dạng tổng quát cho nhiều biến cố.
- Dùng ngôn ngữ xác suất có điều kiện để giải bài toán: một nhà máy có $100$ sản phẩm, trong đó có $5$ cái bị hỏng; chọn lần lượt $3$ sản phẩm, tính xác suất để không có cái nào bị hỏng.


In [ ]:
# Áp dụng chain rule cho ba lần lấy mẫu không hoàn lại: mỗi thừa số là xác suất điều kiện ở một bước.
p_no_defect = (95/100) * (94/99) * (93/98)
print(f"P(no defective item in 3 draws) = {p_no_defect:.6f}")


## 6. Bài 5 - Xét nghiệm y khoa và cập nhật tuần tự

Đây là ví dụ kinh điển cho sự khác biệt giữa độ chính xác của xét nghiệm và posterior cần quan tâm ở cấp cá nhân. Prior rất nhỏ vì bệnh hiếm, nên một kết quả dương tính đơn lẻ chưa đủ mạnh như直 giác thường nghĩ.

### Ý nghĩa của bài tập

Bài xét nghiệm y khoa rất quan trọng vì nó chống lại một trực giác sai rất phổ biến: xét nghiệm có vẻ chính xác không đồng nghĩa với việc một cá nhân dương tính chắc chắn mắc bệnh. Prior ở mức dân số, tức tỷ lệ nền rất thấp, có thể làm posterior sau một lần dương tính vẫn còn khá khiêm tốn.

Phần cập nhật tuần tự còn cho thấy bằng chứng không đứng yên. Mỗi lần xét nghiệm mới lại trở thành dữ liệu mới để điều chỉnh niềm tin cũ, và chính chuỗi cập nhật đó mới phản ánh đúng tư duy Bayesian trong thực hành.


### Đề bài nhắc lại

**Bài 5.** Một căn bệnh có tỷ lệ mắc là $1/10000$. Xét nghiệm có: xác suất dương tính giả $2\%$ và xác suất âm tính giả $1\%$.

- a) Một người test ra dương tính, tính xác suất người đó thực sự mắc bệnh.
- b) Nếu lần thứ hai người này test ra âm tính, xác suất mắc bệnh thay đổi như thế nào?


In [ ]:
# Cập nhật tuần tự cho bài xét nghiệm: posterior sau lần test trước trở thành prior cho lần test sau.
prevalence = 1 / 10000
false_positive = 0.02
false_negative = 0.01
sensitivity = 1 - false_negative
specificity = 1 - false_positive

posterior_after_positive = sensitivity * prevalence / (sensitivity * prevalence + false_positive * (1 - prevalence))
posterior_after_pos_then_neg = false_negative * posterior_after_positive / (
    false_negative * posterior_after_positive + specificity * (1 - posterior_after_positive)
)

print(f"P(disease | +) = {posterior_after_positive:.6f}")
print(f"P(disease | + then -) = {posterior_after_pos_then_neg:.6f}")


### Interpretation

Mặc dù test khá chính xác, xác suất thực sự mắc bệnh sau một lần dương tính chỉ vào khoảng 0.49% vì bệnh quá hiếm. Sau khi có thêm một lần âm tính, posterior giảm xuống rất mạnh. Đây chính là Bayesian update: mỗi kết quả test mới thay đổi niềm tin trước đó.


## 7. Bài 6 - Giải hệ điều kiện xác suất

Ta có thể xem bài này như bài toán “khôi phục bức tranh xác suất” từ những mảnh thông tin điều kiện rời rạc. Khi đề chỉ cho một vài xác suất có điều kiện hay xác suất biên, nhiệm vụ của ta là kiểm tra xem chúng có nhất quán với nhau hay không rồi từ đó suy ra các xác suất còn thiếu.

### Ý nghĩa của bài tập

Ý nghĩa của bài này nằm ở chỗ nhiều bài toán thực tế không cho ta toàn bộ phân phối chung ngay từ đầu. Ta thường chỉ biết một số tỷ lệ cục bộ hoặc các xác suất sau điều kiện. Việc ghép các mảnh đó lại thành một cấu trúc xác suất nhất quán là kỹ năng rất quan trọng trước khi có thể nói đến Bayes hay dự đoán.

Bài cũng nhắc rằng không phải bộ số nào “trông có vẻ hợp lý” cũng tạo thành một mô hình xác suất hợp lệ. Kiểm tra tính nhất quán giữa các điều kiện chính là bước kiểm tra chất lượng mô hình ở mức cơ bản nhất.


### Đề bài nhắc lại

**Bài 6.** Cho các biến cố $A,B,C$ thỏa mãn: $A,C$ độc lập; $B,C$ độc lập; $A,B$ xung khắc; $P(A+B)=2/3$, $P(B+C)=3/4$, và $P(A+B+C)=11/12$. Tính $P(A)$, $P(B)$, và $P(C)$.


In [ ]:
# Bài này giải bằng cách dịch các điều kiện đề bài thành hệ phương trình trên P(A), P(B), P(C).
# Let a=P(A), b=P(B), c=P(C).
# From A,B exclusive: a+b = 2/3.
# From A+B+C = 11/12 and independence with C:
# a+b+c-ac-bc = 11/12 -> 2/3 + c(1-a-b)=11/12 -> c=3/4.
# Then from b+c-bc = 3/4 with c=3/4, we get b=0 and a=2/3.
a = 2/3
b = 0
c = 3/4
print(f"P(A) = {a:.4f}, P(B) = {b:.4f}, P(C) = {c:.4f}")


## 8. Bài 7 - Độc lập có điều kiện

### Phần (a)

Ta gọi đồng xu thứ nhất là đồng xu công bằng HT, đồng xu thứ hai là đồng xu giả HH, và $C$ là biến cố “đồng xu thứ nhất được chọn”. Khi đó:

- $P(A\mid C)=\frac{1}{2}$,
- $P(B\mid C)=\frac{1}{2}$,
- $P(A\cap B\mid C)=\frac{1}{4}$,
- $P(A)=P(B)=\frac{3}{4}$,
- $P(A\cap B)=\frac{5}{8}$.

### Phần (b)

Nếu $A,B$ độc lập có điều kiện theo mọi $C_i$ và $B$ độc lập với từng $C_i$, thì
$$
P(A\cap B)=\sum_i P(A\cap B\mid C_i)P(C_i)=\sum_i P(A\mid C_i)P(B\mid C_i)P(C_i).
$$
Do $P(B\mid C_i)=P(B)$, ta có
$$
P(A\cap B)=P(B)\sum_i P(A\mid C_i)P(C_i)=P(B)P(A),
$$
nên $A,B$ độc lập.

### Ý nghĩa của bài tập

Bài này giúp phân biệt ba khái niệm rất dễ bị lẫn: độc lập thông thường, phụ thuộc do trộn nhiều cơ chế, và độc lập có điều kiện sau khi biết biến ẩn nào đang chi phối quá trình sinh dữ liệu. Hai biến cố có thể phụ thuộc khi nhìn toàn cục, nhưng lại trở nên độc lập khi ta biết được cơ chế nền phía sau.

Đây là ý tưởng xuất hiện lặp đi lặp lại trong mixture models, mô hình phân cấp và causal reasoning. Vì vậy, dù bài toán dùng đồng xu khá đơn giản, trực giác nó xây dựng lại có giá trị rất dài hạn.


### Đề bài nhắc lại

**Bài 7.**

- a) Một hộp có hai đồng xu: một đồng chất $(H,T)$ và một đồng giả $(H,H)$. Chọn ngẫu nhiên một đồng xu rồi tung hai lần. Với các biến cố $A$: lần đầu ra $H$, $B$: lần hai ra $H$, $C$: chọn đồng xu thứ nhất, hãy tính $P(A\mid C)$, $P(B\mid C)$, $P(A.B\mid C)$, $P(A)$, $P(B)$ và $P(A.B)$.
- b) Cho $C_1,\dots,C_k$ là một phân hoạch. Biết $A,B$ độc lập có điều kiện theo từng $C_i$ và $B$ độc lập với từng $C_i$. Chứng minh rằng $A,B$ độc lập.


## 9. Model criticism and reflection

Lab 1 cho thấy xác suất có điều kiện không chỉ là một công thức mà là cách mô tả luồng thông tin. Nếu mô tả thông tin đầu vào mơ hồ hoặc sai cơ chế thu thập dữ liệu, posterior sẽ sai theo. Đây là bài học Bayesian quan trọng nhất của lab: trước khi bấm máy tính, phải nói rõ ta biết điều gì và biết điều đó bằng cách nào.


## 10. Conceptual interpretation questions

1. Vì sao hai mệnh đề “ít nhất một con là nữ” và “chọn ngẫu nhiên một con rồi biết đó là nữ” cho hai posterior khác nhau?  
2. Trong bài xét nghiệm, vì sao một xét nghiệm khá chính xác vẫn có thể cho posterior rất thấp sau khi dương tính?  
3. Ở bài túi bi, điều gì sẽ thay đổi nếu prior chọn túi không còn là đều nhau?  
4. Quy tắc chuỗi giúp ta nhìn bài lấy mẫu không hoàn lại như một quá trình cập nhật tuần tự như thế nào?


## 11. Final takeaway

Tất cả các bài trong Lab 1 đều xoay quanh cùng một ý tưởng: dữ liệu không tự nói lên xác suất, mà phải được đọc thông qua không gian mẫu và cơ chế sinh dữ liệu. Công thức Bayes chỉ là cách viết gọn của quá trình cập nhật này.


## 12. Lecture references

Nếu muốn quay lại đúng phần lý thuyết cho từng cụm bài trong Lab 1, nên đọc lại các lecture sau:

- [Bài 0.1: Nền tảng Xác suất - Ngôn ngữ của Sự Không chắc chắn](../../contents/vi/chapter00/_posts/2025-01-02-00_01_probability_basics.md): **Bài 1** và **Bài 4**: đọc lại cách xác định không gian mẫu, biến cố và xác suất có điều kiện để thấy vì sao biết $B$ hay biết $X+Y=7$ làm đổi bài toán đếm.
- [Bài 1.2: Xác suất như Độ hợp lý - Nền tảng Triết học của Bayesian](../../contents/vi/chapter01/_posts/2025-01-02-01_02_probability_as_plausibility.md): **Bài 3** và **Bài 7**: đọc lại ý “xác suất phụ thuộc vào thông tin đang có” để hiểu vì sao hai phát biểu nghe gần giống nhau vẫn cho posterior khác nhau, và vì sao độc lập có điều kiện phải gắn với biến ẩn.
- [Bài 1.3: Định lý Bayes - Công cụ Cập nhật Niềm tin](../../contents/vi/chapter01/_posts/2025-01-02-01_03_bayes_theorem_posterior.md): **Bài 2** và **Bài 5**: đọc lại sơ đồ prior-likelihood-posterior để thấy cách đi từ viên bi đỏ hoặc kết quả xét nghiệm về nguyên nhân ẩn phía sau dữ liệu.
- [Bài 1.4: Bayesian vs Frequentist - So Sánh và Lựa Chọn](../../contents/vi/chapter01/_posts/2025-01-02-01_04_bayesian_vs_frequentist.md): **Bài 5** và **Bài 6**: đặc biệt cần khi dễ nhầm giữa tỷ lệ nền của quần thể với xác suất hậu nghiệm cho một cá nhân hay một giả thuyết cụ thể.
